In [1]:
import kuzu
from yfiles_jupyter_graphs_for_kuzu import KuzuGraphWidget
from IPython.display import display

In [2]:
print(kuzu.__version__)

0.11.3


In [3]:
db_path = "db/graphiti.kuzu"

In [4]:
cypher = "MATCH (a)-[b]->(c) RETURN * LIMIT 100"
# cypher = """MATCH (a)-[b]->(c)
# WHERE a.group_id = 'e5b08fe6988e'
# RETURN * LIMIT 100"""

In [5]:
# 全局变量，用于后续清理
db = None
conn = None
g = None

### 解除文件锁

In [6]:
# import subprocess
# import os
# import signal
# import time

In [7]:
# def auto_kill_kuzu_blockers():
#     # 关键词：直接匹配文件名，不依赖具体路径，防止相对路径/绝对路径造成的查找失败
#     target_file = "graphiti.kuzu"
    
#     print(f"🗡️ 正在执行: lsof | grep {target_file}")
    
#     try:
#         # 直接使用 shell=True 模拟你在终端的操作
#         # 查找所有包含 graphiti.kuzu 的打开文件记录
#         # 忽略 grep 自身的进程 (grep -v grep)
#         cmd = f"lsof | grep {target_file} | grep -v grep"
        
#         # 运行命令
#         process = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
#         output = process.stdout.strip()

#         if not output:
#             print("✅ 扫描完毕：当前没有进程占用 graphiti.kuzu。")
#             return

#         lines = output.split('\n')
#         current_pid = os.getpid()
#         killed_pids = set()

#         print(f"🔍 发现 {len(lines)} 条占用记录，开始处理...")

#         for line in lines:
#             parts = line.split()
#             # lsof 输出格式通常为: COMMAND PID USER ...
#             # 例如: python3.1 87708 stargazat ...
#             if len(parts) < 2:
#                 continue
                
#             pid_str = parts[1]
#             if not pid_str.isdigit():
#                 continue
            
#             pid = int(pid_str)

#             # 保护机制：防止自杀
#             if pid == current_pid:
#                 continue

#             # 去重，避免同一个进程多次kill报错
#             if pid in killed_pids:
#                 continue

#             print(f"💥 正在查杀僵尸进程 PID: {pid} (持有者: {parts[0]})")
            
#             try:
#                 os.kill(pid, signal.SIGKILL) # 发送 -9 信号
#                 print(f"   └── ✅ PID {pid} 已被强制处决。")
#                 killed_pids.add(pid)
#             except ProcessLookupError:
#                 print(f"   └── ⚠️ PID {pid} 已经不存在了。")
#             except Exception as e:
#                 print(f"   └── ❌ 查杀失败: {e}")

#         if killed_pids:
#             print(f"🎉 清理结束。共处决 {len(killed_pids)} 个进程。")
#             print("⏳ 等待 1 秒释放文件锁...")
#             time.sleep(1)
        
#     except Exception as e:
#         print(f"❌ 脚本执行出错: {e}")

# # --- 直接调用 ---
# auto_kill_kuzu_blockers()

In [8]:
# # 修复代码：只运行一次
# import kuzu
# try:
#     # 1. 不带 read_only=True，默认是读写模式
#     # 这会触发 WAL Checkpoint，把 graphiti.kuzu.wal 合并消失
#     print("🚑 正在尝试以读写模式打开以修复 WAL...")
#     db = kuzu.Database("db/graphiti.kuzu") 
#     conn = kuzu.Connection(db)
#     print("✅ 数据库修复成功！连接已建立。")
    
#     # 2. 这里的 conn 现在就可以用来查询画图了
#     # 或者你可以 del db, del conn 关闭它，然后再用你的只读模式代码
# except Exception as e:
#     print(f"❌ 修复失败: {e}")

### 图显示

In [9]:
try:
    # 1. 以只读模式打开 (如果 Kuzu 版本支持) 或者普通模式
    db = kuzu.Database(db_path, read_only=True)
    conn = kuzu.Connection(db)
    
    # 2. 画图
    print("🎨 正在加载最新记忆图谱...")
    g = KuzuGraphWidget(conn)
    display(g)
    
    # 3. 查询
    g.show_cypher(cypher)

except Exception as e:
    print(f"⚠️ 无法读取数据库，可能智能体正在写入: {e}")
    # 确保异常时也清理资源
    if conn:
        try:
            conn.close()
        except:
            pass
    conn = None
    db = None

🎨 正在加载最新记忆图谱...


GraphWidget(layout=Layout(height='610px', width='100%'))

In [10]:
conn.close()

In [11]:
"""
🔧 清理数据库连接（在停止内核前运行此单元格）
这样可以避免残留的锁文件导致 test.py 执行时出现 segmentation fault
"""
import gc

# 1. 尝试关闭连接
try:
    if 'conn' in locals() or 'conn' in globals():
        conn_obj = locals().get('conn') or globals().get('conn')
        if conn_obj:
            try:
                conn_obj.close()
                print("✅ 连接已关闭")
            except Exception as e:
                print(f"⚠️ 关闭连接时出错: {e}")
    
    if 'g' in locals() or 'g' in globals():
        g_obj = locals().get('g') or globals().get('g')
        if g_obj:
            del g_obj
            print("✅ 图形组件已删除")
    
    if 'conn' in locals() or 'conn' in globals():
        conn_obj = locals().get('conn') or globals().get('conn')
        if conn_obj:
            del conn_obj
    
    if 'db' in locals() or 'db' in globals():
        db_obj = locals().get('db') or globals().get('db')
        if db_obj:
            del db_obj
            print("✅ 数据库对象已删除")
except Exception as e:
    print(f"⚠️ 清理过程中出错: {e}")

# 2. 强制垃圾回收 (关键)
gc.collect()
print("✅ 垃圾回收完成，可以安全停止内核了")

✅ 连接已关闭
✅ 图形组件已删除
✅ 数据库对象已删除
✅ 垃圾回收完成，可以安全停止内核了


In [12]:
# import kuzu
# import os

# # 1. 确定目标文件路径 (根据你的报错日志)
# kuzu_version = kuzu.__version__
# home = os.path.expanduser("~")
# # 注意：路径结构是硬编码的，基于你的报错信息
# fts_path = os.path.join(home, ".kuzu", "extension", kuzu_version, "osx_arm64", "fts", "libfts.kuzu_extension")

# print(f"🔎 正在检查环境...\nKuzu版本: {kuzu_version}")
# print(f"🎯 目标文件路径: {fts_path}")

# # 2. 检查文件是否真的存在
# if os.path.exists(fts_path):
#     print("✅ 文件实际上是存在的！(如果还报错，可能是权限问题或文件损坏)")
# else:
#     print("❌ 文件确实不存在。这就是报错的原因。")
#     print("🚀 正在启动修复程序...")

#     try:
#         # 使用内存模式，避免锁住你的 graphiti.kuzu 文件
#         db = kuzu.Database(":memory:")
#         conn = kuzu.Connection(db)
        
#         print("⏳ 正在执行 'INSTALL fts' (需要联网下载)...")
#         conn.execute("INSTALL fts")
#         print("✅ 安装命令执行完毕。")
        
#         # 再次检查
#         if os.path.exists(fts_path):
#             print("🎉 修复成功！扩展文件已下载。")
#             print("👉 请重启内核 (Kernel -> Restart)，然后运行你的画图代码。")
#         else:
#             print("⚠️ 安装命令没报错，但文件依然没有出现。这通常是网络问题导致下载被拦截。")
            
#     except Exception as e:
#         print(f"💀 安装过程中发生错误: {e}")